# 03 · Gold — star schema for the semantic model

**Goal:** shape the Silver tables into a Kimball-style star schema that Power BI (Import or Direct Lake)
can consume directly, then validate the headline KPIs with Spark SQL so the DAX measures can be
reconciled against a known-good number.

```
                 gold_dim_date
                      │
gold_dim_seller ── gold_fact_opportunity ── gold_dim_account
                      │          │
               gold_dim_stage  gold_dim_product

gold_fact_stage_transition  (grain: one stage change)      → conversion & stage velocity
gold_fact_activity          (grain: one call/email/meeting) → seller productivity
```

Surrogate integer keys on every dimension, date keys as `yyyyMMdd` integers, and a fiscal calendar
(July start by default — change `FISCAL_YEAR_START_MONTH`).

## 0. Configuration

The same notebook runs unchanged on **Databricks** (Free Edition or any workspace), **Microsoft Fabric**
(attached to a Lakehouse) and a **local PySpark + Delta Lake** session. The platform is auto-detected,
or you can force it with the `LAKEHOUSE_PLATFORM` environment variable.

| Platform | Raw files | Tables |
|---|---|---|
| Databricks | Unity Catalog volume `/Volumes/workspace/sales_lakehouse/raw` | `workspace.sales_lakehouse.<table>` |
| Fabric | Lakehouse `Files/raw` | Lakehouse `Tables` (Delta) |
| Local | `../data/raw` | Spark database `sales_lakehouse` |

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
import os
from datetime import datetime, timezone

def _detect_platform() -> str:
    if os.environ.get("LAKEHOUSE_PLATFORM"):
        return os.environ["LAKEHOUSE_PLATFORM"].lower()
    if "DATABRICKS_RUNTIME_VERSION" in os.environ:
        return "databricks"
    if os.path.isdir("/lakehouse/default"):          # Fabric notebook with a default Lakehouse attached
        return "fabric"
    return "local"

PLATFORM = _detect_platform()            # "databricks" | "fabric" | "local"
CATALOG  = "workspace"                   # Databricks Unity Catalog (Free Edition default catalog)
SCHEMA   = "sales_lakehouse"             # Databricks schema / local Spark database

if PLATFORM == "databricks":
    RAW_PATH    = f"/Volumes/{CATALOG}/{SCHEMA}/raw"
    EXPORT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/export"
    def tbl(name: str) -> str:
        return f"{CATALOG}.{SCHEMA}.{name}"
elif PLATFORM == "fabric":
    RAW_PATH    = "Files/raw"
    EXPORT_PATH = "Files/export"
    def tbl(name: str) -> str:
        return name                       # tables live in the attached Lakehouse
else:
    RAW_PATH    = os.path.abspath("../data/raw")
    EXPORT_PATH = os.path.abspath("../lakehouse/export")
    def tbl(name: str) -> str:
        return f"{SCHEMA}.{name}"

try:
    spark  # noqa: F821 - pre-defined on Databricks and Fabric
except NameError:                          # local run: build a Delta-enabled SparkSession
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _builder = (SparkSession.builder.appName("sales-pipeline-lakehouse")
                .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
                .config("spark.sql.warehouse.dir", os.path.abspath("../lakehouse/warehouse")))
    spark = configure_spark_with_delta_pip(_builder).getOrCreate()

if PLATFORM == "databricks":
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    for _vol in ("raw", "export"):
        try:
            spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{_vol}")
        except Exception as _e:              # no privilege → create the volume in Catalog Explorer instead
            print(f"could not create volume '{_vol}': {str(_e).splitlines()[0]}")
elif PLATFORM == "local":
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {SCHEMA}")

RUN_TS = datetime.now(timezone.utc)
print(f"platform={PLATFORM} | raw={RAW_PATH} | export={EXPORT_PATH} | spark={spark.version}")

## 1. Load Silver

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SNAPSHOT_DATE = "2025-06-30"            # as-of date of the dataset, used for open-pipeline ageing
FISCAL_YEAR_START_MONTH = 7             # 7 = July (FY2026 starts 2025-07-01); set 1 for a calendar fiscal year

silver_opps = spark.table(tbl("silver_opportunities"))
silver_hist = spark.table(tbl("silver_opportunity_stage_history"))
silver_acts = spark.table(tbl("silver_activities"))
silver_sell = spark.table(tbl("silver_sellers"))
silver_acct = spark.table(tbl("silver_accounts"))

def add_surrogate_key(df, key_name: str, order_col: str):
    """Deterministic integer surrogate key (dimensions are small, so a single-partition window is fine)."""
    return df.withColumn(key_name, F.row_number().over(Window.orderBy(order_col)))

def date_key(col: str):
    return F.date_format(F.col(col), "yyyyMMdd").cast("int")

## 2. Dimensions

In [ ]:
# --- dim_date -------------------------------------------------------------------
fiscal_year = (F.year("date") if FISCAL_YEAR_START_MONTH == 1
               else F.when(F.month("date") >= FISCAL_YEAR_START_MONTH, F.year("date") + 1).otherwise(F.year("date")))

dim_date = (spark.sql("SELECT explode(sequence(to_date('2021-01-01'), to_date('2026-12-31'), interval 1 day)) AS date")
    .withColumn("date_key", date_key("date"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("month_short", F.date_format("date", "MMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("fiscal_year", fiscal_year)
    .withColumn("fiscal_quarter", (F.floor(((F.month("date") - F.lit(FISCAL_YEAR_START_MONTH) + 12) % 12) / 3) + 1).cast("int"))
    .withColumn("fiscal_period", F.concat(F.lit("FY"), F.col("fiscal_year").cast("string"), F.lit(" Q"), F.col("fiscal_quarter").cast("string")))
    .select("date_key", "date", "year", "quarter", "month", "month_name", "month_short", "year_month",
            "day", "day_of_week", "day_name", "is_weekend", "week_of_year",
            "fiscal_year", "fiscal_quarter", "fiscal_period"))

# --- dim_seller / dim_account ---------------------------------------------------
dim_seller = add_surrogate_key(silver_sell.drop("_silver_ts"), "seller_key", "seller_id") \
               .select("seller_key", "seller_id", "seller_name", "region", "segment", "team",
                       "manager_name", "hire_date", "quota_annual", "is_active")

dim_account = add_surrogate_key(silver_acct, "account_key", "account_id") \
                .select("account_key", "account_id", "account_name", "industry", "country", "region",
                        "segment", "employee_band", "created_date", "owner_seller_id")

# --- dim_product ----------------------------------------------------------------
PRODUCT_FAMILY = {"Cloud Suite": "Platform", "Analytics Platform": "Data & AI", "Security Bundle": "Security",
                  "Collaboration Tools": "Productivity", "Support Plan": "Services"}
family_map = F.create_map(*[F.lit(x) for kv in PRODUCT_FAMILY.items() for x in kv])
dim_product = add_surrogate_key(
    silver_opps.select("product").distinct()
               .withColumn("product_family", F.coalesce(family_map[F.col("product")], F.lit("Other"))),
    "product_key", "product").select("product_key", "product", "product_family")

# --- dim_stage (static conformed dimension) -------------------------------------
stage_rows = [
    ("Prospecting",   1, 0.10, False, False, "Open"),
    ("Qualification", 2, 0.25, False, False, "Open"),
    ("Proposal",      3, 0.50, False, False, "Open"),
    ("Negotiation",   4, 0.75, False, False, "Open"),
    ("Closed Won",    5, 1.00, True,  True,  "Won"),
    ("Closed Lost",   6, 0.00, True,  False, "Lost"),
]
dim_stage = (spark.createDataFrame(stage_rows,
                "stage_name string, stage_order int, default_probability double, is_closed boolean, is_won boolean, stage_group string")
             .withColumn("stage_key", F.col("stage_order"))
             .select("stage_key", "stage_name", "stage_order", "stage_group", "default_probability", "is_closed", "is_won"))

for name, df in {"dim_date": dim_date, "dim_seller": dim_seller, "dim_account": dim_account,
                 "dim_product": dim_product, "dim_stage": dim_stage}.items():
    print(f"{name:<12} {df.count():>6,} rows")

## 3. Facts

In [ ]:
# --- fact_opportunity (grain: one row per opportunity) --------------------------
acts_per_opp = (silver_acts.groupBy("opportunity_id")
                .agg(F.count("*").alias("activity_count"), F.sum("duration_minutes").alias("activity_minutes")))

fact_opportunity = (silver_opps
    .join(dim_account.select("account_key", "account_id"), "account_id", "left")
    .join(dim_seller.select("seller_key", "seller_id"), "seller_id", "left")
    .join(dim_product.select("product_key", "product"), "product", "left")
    .join(dim_stage.select("stage_key", F.col("stage_name").alias("stage")), "stage", "left")
    .join(acts_per_opp, "opportunity_id", "left")
    .withColumn("created_date_key", date_key("created_date"))
    .withColumn("expected_close_date_key", date_key("expected_close_date"))
    .withColumn("actual_close_date_key", date_key("actual_close_date"))
    .withColumn("days_open", F.when(~F.col("is_closed"), F.datediff(F.lit(SNAPSHOT_DATE).cast("date"), F.col("created_date"))))
    .withColumn("activity_count", F.coalesce(F.col("activity_count"), F.lit(0)))
    .withColumn("activity_minutes", F.coalesce(F.col("activity_minutes"), F.lit(0)))
    .select("opportunity_id", "opportunity_name", "account_key", "seller_key", "product_key", "stage_key",
            "created_date", "created_date_key", "expected_close_date", "expected_close_date_key",
            "actual_close_date", "actual_close_date_key",
            "lead_source", "currency", "amount", "probability", "weighted_amount", "amount_missing",
            "is_closed", "is_won", "sales_cycle_days", "days_open", "activity_count", "activity_minutes"))

# --- fact_stage_transition (grain: one stage change) ----------------------------
w_opp = Window.partitionBy("opportunity_id").orderBy("changed_at")
fact_stage_transition = (silver_hist
    .withColumn("prev_changed_at", F.lag("changed_at").over(w_opp))
    .withColumn("days_in_from_stage", F.round((F.unix_timestamp("changed_at") - F.unix_timestamp("prev_changed_at")) / 86400.0, 1))
    .join(dim_stage.select(F.col("stage_key").alias("from_stage_key"), F.col("stage_name").alias("from_stage")), "from_stage", "left")
    .join(dim_stage.select(F.col("stage_key").alias("to_stage_key"), F.col("stage_name").alias("to_stage")), "to_stage", "left")
    .join(fact_opportunity.select("opportunity_id", "seller_key", "account_key", "product_key"), "opportunity_id", "left")
    .withColumn("changed_date_key", date_key("changed_at"))
    .select("history_id", "opportunity_id", "seller_key", "account_key", "product_key",
            "from_stage_key", "to_stage_key", "from_stage", "to_stage",
            "changed_at", "changed_date_key", "days_in_from_stage"))

# --- fact_activity (grain: one activity) ----------------------------------------
fact_activity = (silver_acts
    .join(fact_opportunity.select("opportunity_id", "account_key", "product_key"), "opportunity_id", "left")
    .join(dim_seller.select("seller_key", "seller_id"), "seller_id", "left")
    .withColumn("activity_date_key", date_key("activity_ts"))
    .select("activity_id", "opportunity_id", "seller_key", "account_key", "product_key",
            "activity_type", "activity_ts", "activity_date_key", "duration_minutes"))

for name, df in {"fact_opportunity": fact_opportunity, "fact_stage_transition": fact_stage_transition,
                 "fact_activity": fact_activity}.items():
    print(f"{name:<22} {df.count():>8,} rows")

## 4. Write the Gold layer

In [ ]:
GOLD_TABLES = {
    "gold_dim_date": dim_date,
    "gold_dim_seller": dim_seller,
    "gold_dim_account": dim_account,
    "gold_dim_product": dim_product,
    "gold_dim_stage": dim_stage,
    "gold_fact_opportunity": fact_opportunity,
    "gold_fact_stage_transition": fact_stage_transition,
    "gold_fact_activity": fact_activity,
}
for name, df in GOLD_TABLES.items():
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl(name)))
    print(f"{name:<28} {spark.table(tbl(name)).count():>8,} rows")

## 5. KPI reconciliation with Spark SQL

These are the same numbers the DAX measures in `model/dax_measures.md` must return — use them to
validate the semantic model.

In [ ]:
spark.sql(f"""
SELECT
  ROUND(SUM(CASE WHEN NOT is_closed THEN amount END) / 1e6, 2)           AS open_pipeline_musd,
  ROUND(SUM(CASE WHEN NOT is_closed THEN weighted_amount END) / 1e6, 2)  AS weighted_pipeline_musd,
  ROUND(SUM(CASE WHEN is_won THEN amount END) / 1e6, 2)                  AS won_revenue_musd,
  ROUND(SUM(CASE WHEN is_won THEN 1 ELSE 0 END)
        / SUM(CASE WHEN is_closed THEN 1 ELSE 0 END), 3)                 AS win_rate,
  ROUND(AVG(CASE WHEN is_won THEN sales_cycle_days END), 1)              AS avg_won_cycle_days,
  COUNT(*)                                                                AS opportunities,
  SUM(CASE WHEN NOT is_closed THEN 1 ELSE 0 END)                          AS open_opportunities
FROM {tbl('gold_fact_opportunity')}
""").show()

spark.sql(f"""
SELECT s.region, s.segment,
       COUNT(*)                                                   AS deals,
       ROUND(SUM(CASE WHEN f.is_won THEN 1 ELSE 0 END)
             / NULLIF(SUM(CASE WHEN f.is_closed THEN 1 ELSE 0 END), 0), 3) AS win_rate,
       ROUND(SUM(CASE WHEN f.is_won THEN f.amount END) / 1e6, 2)  AS won_musd,
       ROUND(AVG(CASE WHEN f.is_won THEN f.sales_cycle_days END), 1) AS avg_won_cycle_days
FROM {tbl('gold_fact_opportunity')} f
JOIN {tbl('gold_dim_seller')} s ON f.seller_key = s.seller_key
GROUP BY s.region, s.segment
ORDER BY s.region, s.segment
""").show()

spark.sql(f"""
SELECT from_stage, to_stage, COUNT(*) AS transitions, ROUND(AVG(days_in_from_stage), 1) AS avg_days_in_from_stage
FROM {tbl('gold_fact_stage_transition')}
WHERE from_stage IS NOT NULL
GROUP BY from_stage, to_stage
ORDER BY from_stage, to_stage
""").show(truncate=False)

## 6. Export for Power BI Desktop (Import mode)

Without a Fabric capacity the report uses Import mode, so the Gold tables are exported as CSV
(one file per table) and Parquet. In Fabric the same tables are read by a **Direct Lake** model — no export needed.

Power BI Desktop → *Get data* → **Folder** → point at `export/csv/<table>` (or use the Databricks connector).

In [ ]:
for name in GOLD_TABLES:
    (spark.table(tbl(name)).coalesce(1).write.mode("overwrite")
          .option("header", "true").csv(f"{EXPORT_PATH}/csv/{name}"))
    (spark.table(tbl(name)).write.mode("overwrite").parquet(f"{EXPORT_PATH}/parquet/{name}"))
print("Gold tables exported to", EXPORT_PATH)

✅ **Gold complete.** Build the semantic model with `model/dax_measures.md` and the report with `model/report_layout.md`.